# 02 Baseline ASR Evaluation

Evaluasi tiga model ASR pre-trained (tanpa fine-tuning) pada rekaman presentasi mahasiswa Filkom UB.

**Models:** faster-whisper medium · Wav2Vec 2.0 XLSR-53 Indonesian · Moonshine base
**Metrik:** WER · CER · RTF · TTER (Tech Term Error Rate)

## 1. Install Dependencies

In [ ]:
# !pip install -q faster-whisper transformers torch torchaudio
# !pip install -q jiwer librosa soundfile imageio-ffmpeg tqdm pandas matplotlib
# !pip install -q useful-moonshine

## 2. Import & Konfigurasi

In [1]:
import os, time, json, warnings, re, subprocess
import numpy as np
import librosa
import torch
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from jiwer import wer, cer
import imageio_ffmpeg

warnings.filterwarnings('ignore')

# Path Config 
BASE_DIR   = Path('..') 
DATA_DIR   = BASE_DIR / 'data' / 'raw' / 'presentasi'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device     : {DEVICE}')
print(f'Data dir   : {DATA_DIR.resolve()}')
if DEVICE == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device     : cuda
Data dir   : C:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\raw\presentasi
GPU        : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM       : 6.4 GB


## 3. Load Data

In [2]:
AUDIO_EXTS = ['.wav', '.WAV', '.mp4', '.m4a', '.ogg', '.mp3', '.flac']
all_audio = [f for f in DATA_DIR.iterdir()
             if f.suffix in AUDIO_EXTS and 'audio' in f.name.lower()]
all_audio = sorted(all_audio)

sessions = []
for af in all_audio:
    gt_name = af.name.replace('audio', 'teks')
    gt_name = re.sub(r'\.(wav|WAV|mp4|m4a|ogg|mp3|flac)$', '.txt', gt_name)
    gt_path = DATA_DIR / gt_name
    
    if gt_path.exists():
        sessions.append({'audio': af, 'gt': gt_path})
        print(f'✅  {af.name:35s} → {gt_path.name}')
    else:
        print(f'⚠️  {af.name:35s} → ground truth tidak ditemukan, skip')

assert sessions, 'Tidak ada pasangan audio+ground truth yang ditemukan!'
print(f'\n→ {len(sessions)} sesi siap dievaluasi')

✅  presentasi1-audio.WAV               → presentasi1-teks.txt
✅  presentasi2-audio.WAV               → presentasi2-teks.txt
✅  presentasi3-audio.wav               → presentasi3-teks.txt

→ 3 sesi siap dievaluasi


## 4. Preprocessing, Konversi ke WAV 16kHz

In [3]:
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def convert_to_wav16k(audio_path: Path, output_dir: Path) -> Path:
    out = output_dir / (audio_path.stem + '_16k.wav')
    if out.exists():
        return out
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
    cmd = [ffmpeg, '-i', str(audio_path),
           '-ac', '1', '-ar', str(SAMPLE_RATE),
           str(out), '-y', '-loglevel', 'error']
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'ffmpeg error: {r.stderr}')
    return out

print('Memproses file audio...')
for s in sessions:
    s['wav']          = convert_to_wav16k(s['audio'], OUTPUT_DIR)
    s['audio_array'], _ = librosa.load(str(s['wav']), sr=SAMPLE_RATE, mono=True)
    s['duration']     = len(s['audio_array']) / SAMPLE_RATE
    s['gt_raw']       = s['gt'].read_text(encoding='utf-8')
    s['gt_norm']      = normalize_text(s['gt_raw'])
    print(f'  {s["audio"].name}: {s["duration"]:.1f}s '
          f'({s["duration"]/60:.1f} mnt) | GT: {len(s["gt_norm"].split())} kata')

Memproses file audio...
  presentasi1-audio.WAV: 90.7s (1.5 mnt) | GT: 193 kata
  presentasi2-audio.WAV: 122.7s (2.0 mnt) | GT: 270 kata
  presentasi3-audio.wav: 233.2s (3.9 mnt) | GT: 474 kata


## 5. Helper Evaluasi & Tech Term Analysis

In [4]:
TECH_TERMS = [
    # CS Fundamentals
    'algoritma', 'fungsi', 'variabel', 'array', 'rekursi', 'iterasi',
    'stack', 'queue', 'tree', 'graph', 'node', 'edge', 'branch',
    # OOP (presentasi2: Class Diagram, Abstract, Constructor)
    'class', 'abstract', 'constructor', 'method', 'attribute', 'interface',
    'inheritance', 'override', 'polymorphism', 'encapsulation', 'instansiasi',
    'object', 'diagram',
    # Graph & Algorithm (presentasi3: Graph Coloring, Backtracking, DFS)
    'backtracking', 'coloring', 'pewarnaan', 'simpul', 'dead node',
    'depth first search', 'dfs', 'eliminasi',
    # CV/ML (presentasi1: CNN, preprocessing, feature extraction)
    'preprocessing', 'classification', 'segmentasi', 'ekstraksi',
    'dataset', 'convolutional', 'cnn', 'feature extraction',
    'smoothing', 'resize', 'tekstur',
    # General CS
    'pipeline', 'implementasi', 'sistem', 'parameter', 'output', 'input',
]

results_all = {}

def evaluate(model_name, hypothesis, gt_raw, inference_time, duration):
    hyp = normalize_text(hypothesis)
    ref = normalize_text(gt_raw)

    word_err = wer(ref, hyp)
    char_err = cer(ref, hyp)
    rtf      = inference_time / duration

    # TTER: berapa % kata teknis yang miss
    tech_in_ref  = [t for t in TECH_TERMS if t in ref]
    tech_correct = [t for t in tech_in_ref if t in hyp]
    tech_missed  = [t for t in tech_in_ref if t not in hyp]
    tter = (len(tech_missed) / len(tech_in_ref) * 100) if tech_in_ref else 0.0

    return {
        'model'         : model_name,
        'wer'           : round(word_err * 100, 2),
        'cer'           : round(char_err * 100, 2),
        'rtf'           : round(rtf, 4),
        'tter'          : round(tter, 2),
        'inference_sec' : round(inference_time, 2),
        'duration_sec'  : round(duration, 2),
        'tech_in_ref'   : tech_in_ref,
        'tech_correct'  : tech_correct,
        'tech_missed'   : tech_missed,
        'transcription' : hypothesis,
    }

def print_result(res, session_name=''):
    label = f'{res["model"]}' + (f' | {session_name}' if session_name else '')
    print(f'\n{"="*58}')
    print(f'  {label}')
    print(f'  WER  : {res["wer"]:6.2f}%  {"✅" if res["wer"] < 20 else "⚠️"}')
    print(f'  CER  : {res["cer"]:6.2f}%')
    print(f'  RTF  : {res["rtf"]:6.4f}   {"✅ real-time" if res["rtf"] < 1 else "⚠️ lambat"}')
    print(f'  TTER : {res["tter"]:6.2f}%  (tech term miss rate)')
    print(f'  Teknis benar  : {res["tech_correct"]}')
    print(f'  Teknis terlewat: {res["tech_missed"]}')
    print(f'{"─"*58}')
    print(' '.join(res['transcription'].split()[:120]))

## 6. Model A faster-whisper medium

In [26]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [22]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel('medium', device=DEVICE, compute_type='int8_float16')
print('faster-whisper medium loaded.')

faster-whisper medium loaded.


In [27]:
results_all['faster_whisper_medium'] = []

for s in sessions:
    t0 = time.time()
    segments, _ = whisper_model.transcribe(
        str(s['wav']),
        language='id',
        beam_size=5,
        condition_on_previous_text=True,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=300),
    )
    text    = ' '.join(seg.text.strip() for seg in segments)
    elapsed = time.time() - t0

    res = evaluate('faster-whisper medium', text, s['gt_raw'], elapsed, s['duration'])
    results_all['faster_whisper_medium'].append(res)
    print_result(res, s['audio'].name)
    (OUTPUT_DIR / f'whisper_medium_{s["audio"].stem}.txt').write_text(text, encoding='utf-8')

print('\n✅ Whisper selesai.')


  faster-whisper medium | presentasi1-audio.WAV
  WER  :  69.95%  ⚠️
  CER  :  40.14%
  RTF  : 0.3300   ✅ real-time
  TTER :  75.00%  (tech term miss rate)
  Teknis benar  : ['preprocessing', 'pipeline', 'sistem']
  Teknis terlewat: ['class', 'classification', 'segmentasi', 'dataset', 'cnn', 'feature extraction', 'smoothing', 'resize', 'tekstur']
──────────────────────────────────────────────────────────
Pada tahun 2001, semua perempangan dari kelas pengolahan cita juga redi gratis komputer Dengan ini 2351-502-0111 Dan juga di sini saya akui menerima pesan perempangan saya Yaitu Dujia Hemaboni yang ini 2351-502-0111-012 Ini 3800 jahitan terkait dengan progress project akhir PCE di UK Yang kemarin pagi-pagi mabuk-mabuk, kemarin klasifikasi Yang namanya mengklasifikasi, kelas organik dan antarganik Untuk deskripsi project ini semuanya lebih lanjut Kami akan memberikan sistem klasifikasi bersama Untuk menjadi dua kategori, yang itu organik dan antarganik Yang ini menghubungkan kempek, mi

RuntimeError: CUDA failed with error out of memory

## 7. Model B faster-whisper large-v3-turbo

In [ ]:
whisper_turbo = WhisperModel('large-v3-turbo', device=DEVICE, compute_type='int8')
print('faster-whisper large-v3-turbo loaded.')

faster-whisper large-v3-turbo loaded.


In [ ]:
results_all['faster_whisper_turbo'] = []

for s in sessions:
    t0 = time.time()
    segments, _ = whisper_turbo.transcribe(
        str(s['wav']),
        language='id',
        beam_size=5,
        condition_on_previous_text=True,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=300),
    )
    text = ' '.join(seg.text.strip() for seg in segments)
    elapsed = time.time() - t0

    res = evaluate('faster-whisper large-v3-turbo', text, s['gt_raw'], elapsed, s['duration'])
    results_all['faster_whisper_turbo'].append(res)
    print_result(res, s['audio'].name)
    (OUTPUT_DIR / f'whisper_turbo_{s["audio"].stem}.txt').write_text(text, encoding='utf-8')

print('\n✅ Whisper turbo selesai.')


  faster-whisper large-v3-turbo | presentasi1-audio.WAV
  WER  : 100.00%  ⚠️
  CER  :  97.33%
  RTF  : 0.0400   ✅ real-time
  TTER : 100.00%  (tech term miss rate)
  Teknis benar  : []
  Teknis terlewat: ['class', 'preprocessing', 'classification', 'segmentasi', 'dataset', 'cnn', 'feature extraction', 'smoothing', 'resize', 'tekstur', 'pipeline', 'sistem']
──────────────────────────────────────────────────────────
Terima kasih. Terima kasih. Terima kasih.

  faster-whisper large-v3-turbo | presentasi2-audio.WAV
  WER  :  32.96%  ⚠️
  CER  :  15.37%
  RTF  : 0.0709   ✅ real-time
  TTER :  77.78%  (tech term miss rate)
  Teknis benar  : ['interface', 'override']
  Teknis terlewat: ['class', 'abstract', 'constructor', 'method', 'attribute', 'diagram', 'implementasi']
──────────────────────────────────────────────────────────
Di sini saya akan menjelaskan bersama dengan pekan saya, yaitu Nui Cahaya Malang, dengan yang 2351-502-601-1103. Ya, mungkin ini akan langsung saya bawa yang pertama

## 8. Model B Wav2Vec 2.0 XLSR-53 Indonesian

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

W2V_ID = 'indonesian-nlp/wav2vec2-large-xlsr-indonesian'
w2v_processor = Wav2Vec2Processor.from_pretrained(W2V_ID)
w2v_model     = Wav2Vec2ForCTC.from_pretrained(W2V_ID).to(DEVICE)
w2v_model.eval()
print('Wav2Vec 2.0 loaded.')

Wav2Vec 2.0 loaded.


In [ ]:
def wav2vec_infer(audio, chunk_sec=25, overlap_sec=1):
    chunk_s, overlap_s = chunk_sec * SAMPLE_RATE, overlap_sec * SAMPLE_RATE
    step, segs = chunk_s - overlap_s, []
    for start in range(0, len(audio), step):
        chunk = audio[start : start + chunk_s]
        if len(chunk) < 1600:
            continue
        inp = w2v_processor(chunk, sampling_rate=SAMPLE_RATE,
                            return_tensors='pt', padding=True).input_values.to(DEVICE)
        with torch.no_grad():
            logits = w2v_model(inp).logits
        segs.append(w2v_processor.batch_decode(torch.argmax(logits, -1))[0].strip())
    return ' '.join(segs)

results_all['wav2vec2_xlsr'] = []

for s in sessions:
    t0   = time.time()
    text = wav2vec_infer(s['audio_array'])
    elapsed = time.time() - t0

    res = evaluate('Wav2Vec 2.0 XLSR-53', text, s['gt_raw'], elapsed, s['duration'])
    results_all['wav2vec2_xlsr'].append(res)
    print_result(res, s['audio'].name)
    (OUTPUT_DIR / f'wav2vec_{s["audio"].stem}.txt').write_text(text, encoding='utf-8')

print('\n✅ Wav2Vec selesai.')


  Wav2Vec 2.0 XLSR-53 | presentasi1-audio.WAV
  WER  :  96.89%  ⚠️
  CER  :  68.21%
  RTF  : 0.0456   ✅ real-time
  TTER : 100.00%  (tech term miss rate)
  Teknis benar  : []
  Teknis terlewat: ['class', 'preprocessing', 'classification', 'segmentasi', 'dataset', 'cnn', 'feature extraction', 'smoothing', 'resize', 'tekstur', 'pipeline', 'sistem']
──────────────────────────────────────────────────────────
kamkahu buan tuusuapergimakan saahdisan sudaadadari kelas tahanolag kaang kida iara kpaa siagita dengan ida didir lasi watetupahotinutsua ta salu suat idi tiga pendaaya kain mtida ke pemapanayaayahkudisioaapakantikua di dua kiapasakumkipatntuan tahn an sata tutusaltlahal tuuka anakantaan kemalam ilasanakai dita imabrorsbantitakhad peca herpinsatayan kemmarieaman diibui sai sekosi payadiriyang lagiiy amadikasi dikasiaraapor gada idiaa penaainkaasitu dispiimacaisarusa bilai kuuanaatsinamiyang kakagkalajiapasiaiasisaemahdaaadiindua ka tahodi i kekardari danlaisindilamu kugan iaa ekai ini

## 9. Model C MMS (Meta) facebook/mms-300m

In [16]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2FeatureExtractor

MMS_ID = 'facebook/mms-1b-fl102'
mms_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MMS_ID)
mms_model = Wav2Vec2ForCTC.from_pretrained(MMS_ID, target_lang='ind', ignore_mismatched_sizes=True).to(DEVICE)
mms_model.eval()
tokenizer_ind = mms_model.load_adapter('ind')
print('MMS 1B loaded.')

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-fl102 and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([78]) in the checkpoint and torch.Size([71]) in the model instantiated
- lm_head.weight: found shape torch.Size([78, 1280]) in the checkpoint and torch.Size([71, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


MMS 1B loaded.


In [19]:
from transformers import Wav2Vec2CTCTokenizer

mms_tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(MMS_ID, target_lang='ind')

In [20]:
def mms_transcribe(audio, chunk_sec=25, overlap_sec=1):
    chunk_s, overlap_s = chunk_sec * SAMPLE_RATE, overlap_sec * SAMPLE_RATE
    step, segs = chunk_s - overlap_s, []
    for start in range(0, len(audio), step):
        chunk = audio[start : start + chunk_s]
        if len(chunk) < 1600:
            continue
        inp = mms_feature_extractor(
            chunk, sampling_rate=SAMPLE_RATE,
            return_tensors='pt', padding=True
        ).input_values.to(DEVICE)
        with torch.no_grad():
            logits = mms_model(inp).logits
        pred_ids = torch.argmax(logits, dim=-1)
        segs.append(mms_tokenizer.decode(pred_ids[0]))
    return ' '.join(segs)

results_all['mms_1b'] = []

for s in sessions:
    t0   = time.time()
    text = mms_transcribe(s['audio_array'])
    elapsed = time.time() - t0

    res = evaluate('MMS 1B fl102', text, s['gt_raw'], elapsed, s['duration'])
    results_all['mms_1b'].append(res)
    print_result(res, s['audio'].name)

print('\n✅ MMS selesai.')


  MMS 1B fl102 | presentasi1-audio.WAV
  WER  :  90.67%  ⚠️
  CER  :  53.54%
  RTF  : 0.0598   ✅ real-time
  TTER : 100.00%  (tech term miss rate)
  Teknis benar  : []
  Teknis terlewat: ['class', 'preprocessing', 'classification', 'segmentasi', 'dataset', 'cnn', 'feature extraction', 'smoothing', 'resize', 'tekstur', 'pipeline', 'sistem']
──────────────────────────────────────────────────────────
meruaalagi saa eramakan jaa aid racila heriklas dan cita dinal eadugatis komputer dengan n3515113i saya aka meliaka bersamarakantaya yaitu dicahaya baai g2351611 ant etigga leammelasdan perkai dengan progres pojet akhir pcd rbwekan omerin uupay ngambi bros padis klasivaang yaitu yang adunya manu klasifikasikan organi ua adeskripsi ajei nablajon sinami ankar ada misistr klasifikasi sanba yang intuamenjadi du kategori itu organik denariihubuhgan danteknik becere juga feka letasatnya itu sendi an miengambietasa dadi egambirdase dari gageid weisecafeitisien data misi juga sudahang seter bagimen 

## 10. Summary Rata-rata per Model

In [21]:
rows = []
for key, res_list in results_all.items():
    if not res_list:
        continue
    rows.append({
        'Model'       : res_list[0]['model'],
        'WER avg (%)'  : round(np.mean([r['wer']  for r in res_list]), 2),
        'CER avg (%)'  : round(np.mean([r['cer']  for r in res_list]), 2),
        'RTF avg'      : round(np.mean([r['rtf']  for r in res_list]), 4),
        'TTER avg (%)'  : round(np.mean([r['tter'] for r in res_list]), 2),
        'Sessions'     : len(res_list),
    })

df = pd.DataFrame(rows).sort_values('WER avg (%)')
print('\n SUMMARY BASELINE')
print('─' * 65)
print(df.to_string(index=False))
print('─' * 65)
print('WER target : < 15% (umum) / < 20% (terminologi teknis)')
print('RTF target : < 1.0 layak untuk streaming')


 SUMMARY BASELINE
─────────────────────────────────────────────────────────────────
         Model  WER avg (%)  CER avg (%)  RTF avg  TTER avg (%)  Sessions
  MMS 1B fl102        72.74        37.62   0.1032         88.89         3
Moonshine base       148.70        76.07   0.7916        100.00         1
─────────────────────────────────────────────────────────────────
WER target : < 15% (umum) / < 20% (terminologi teknis)
RTF target : < 1.0 layak untuk streaming


## 11. Visualisasi

In [ ]:
if rows:
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    fig.suptitle('Baseline ASR Evaluation Presentasi Dataset\n'
                 '(faster-whisper medium vs Wav2Vec 2.0 XLSR-53 vs Moonshine base)',
                 fontsize=12, fontweight='bold')

    models = df['Model'].tolist()
    metrics = [
        (df['WER avg (%)'].tolist(),  'WER avg (%)',  'WER lebih kecil lebih baik',  15,  'Target 15%'),
        (df['CER avg (%)'].tolist(),  'CER avg (%)',  'CER lebih kecil lebih baik',  None, None),
        (df['RTF avg'].tolist(),      'RTF',          'RTF < 1.0 untuk streaming',   1.0, 'Real-time boundary'),
        (df['TTER avg (%)'].tolist(), 'TTER avg (%)', 'TTER tech term miss rate',    None, None),
    ]
    palette = ['#2563EB', '#16A34A', '#D97706'][:len(models)]

    for ax, (vals, xlabel, title, thresh, tlabel) in zip(axes, metrics):
        bars = ax.barh(models, vals, color=palette, edgecolor='white', height=0.5)
        if thresh:
            ax.axvline(thresh, color='gray', ls='--', lw=1.2, label=tlabel)
            ax.legend(fontsize=8)
        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.set_xlabel(xlabel)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_width() + max(vals)*0.01,
                    bar.get_y() + bar.get_height()/2,
                    f'{v}', va='center', fontsize=9)
        ax.grid(True, axis='x', alpha=0.3)
        ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    out_fig = OUTPUT_DIR / 'baseline_comparison.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot disimpan: {out_fig}')

## 12. Simpan & Kesimpulan

In [ ]:
# Simpan JSON
(OUTPUT_DIR / 'baseline_results.json').write_text(
    json.dumps(results_all, ensure_ascii=False, indent=2), encoding='utf-8')
print('Hasil disimpan ke baseline_results.json')

# Kesimpulan otomatis
if rows:
    best_wer  = min(rows, key=lambda x: x['WER avg (%)'])
    best_rtf  = min(rows, key=lambda x: x['RTF avg'])
    best_tter = min(rows, key=lambda x: x['TTER avg (%)'])
    print()
    print('━' * 58)
    print('📋 KESIMPULAN BASELINE')
    print('━' * 58)
    print(f'✅ Akurasi terbaik  (WER↓)  : {best_wer["Model"]}  →  {best_wer["WER avg (%)"]:.2f}%')
    print(f'⚡ Kecepatan terbaik (RTF↓) : {best_rtf["Model"]}  →  RTF {best_rtf["RTF avg"]:.4f}')
    print(f'🔤 Tech term terbaik (TTER↓): {best_tter["Model"]}  →  {best_tter["TTER avg (%)"]:.2f}%')
    print()
    print('Next steps:')
    print('  → Jika WER > 15%: fine-tune di Kaggle (T4/A100) dengan data Filkom UB')
    print('  → Prioritaskan model dengan TTER terendah untuk domain teknis')
    print('  → Target setelah fine-tune: WER < 15%, TTER < 10%')
    print('━' * 58)